## Setup: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate
!pip install -q git+https://github.com/openai/CLIP.git
print("✓ Dependencies installed")

## Import Libraries

In [ ]:
import torch
from typing import List, Dict, Tuple
import numpy as np
import clip
from PIL import Image
import json
import time
from datetime import datetime
from diffusers import PixArtAlphaPipeline
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## DrawBench Prompts (50 Total)

In [ ]:
DRAWBENCH_PROMPTS = [
    # === COLOR BINDING (20 prompts) ===
    {"id": "color_01", "prompt": "a red cube on top of a blue cube", "attributes": ["red cube", "blue cube"]},
    {"id": "color_02", "prompt": "a green apple and a red apple", "attributes": ["green apple", "red apple"]},
    {"id": "color_03", "prompt": "a yellow car and a blue car", "attributes": ["yellow car", "blue car"]},
    {"id": "color_04", "prompt": "a purple cat sitting on an orange couch", "attributes": ["purple cat", "orange couch"]},
    {"id": "color_05", "prompt": "a pink elephant next to a green tree", "attributes": ["pink elephant", "green tree"]},
    {"id": "color_06", "prompt": "a silver robot holding a golden ball", "attributes": ["silver robot", "golden ball"]},
    {"id": "color_07", "prompt": "a white dog wearing a red collar", "attributes": ["white dog", "red collar"]},
    {"id": "color_08", "prompt": "a black cat with blue eyes", "attributes": ["black cat", "blue eyes"]},
    {"id": "color_09", "prompt": "a brown horse next to a white fence", "attributes": ["brown horse", "white fence"]},
    {"id": "color_10", "prompt": "a red bird on a yellow branch", "attributes": ["red bird", "yellow branch"]},
    {"id": "color_11", "prompt": "a blue butterfly on a pink flower", "attributes": ["blue butterfly", "pink flower"]},
    {"id": "color_12", "prompt": "a green frog on a brown log", "attributes": ["green frog", "brown log"]},
    {"id": "color_13", "prompt": "a purple umbrella next to a yellow raincoat", "attributes": ["purple umbrella", "yellow raincoat"]},
    {"id": "color_14", "prompt": "an orange basketball and a white basketball", "attributes": ["orange basketball", "white basketball"]},
    {"id": "color_15", "prompt": "a red strawberry on a white plate", "attributes": ["red strawberry", "white plate"]},
    {"id": "color_16", "prompt": "a golden crown on a red pillow", "attributes": ["golden crown", "red pillow"]},
    {"id": "color_17", "prompt": "a silver car parked next to a golden bicycle", "attributes": ["silver car", "golden bicycle"]},
    {"id": "color_18", "prompt": "a cyan teapot and a magenta cup", "attributes": ["cyan teapot", "magenta cup"]},
    {"id": "color_19", "prompt": "a navy blue boat on a turquoise sea", "attributes": ["navy blue boat", "turquoise sea"]},
    {"id": "color_20", "prompt": "a lime green lizard on a coral rock", "attributes": ["lime green lizard", "coral rock"]},

    # === MULTI-OBJECT COMPOSITION (15 prompts) ===
    {"id": "comp_01", "prompt": "a cat and a dog sitting together", "attributes": ["cat", "dog"]},
    {"id": "comp_02", "prompt": "a book on a table next to a lamp", "attributes": ["book", "table", "lamp"]},
    {"id": "comp_03", "prompt": "a cup of coffee and a croissant on a plate", "attributes": ["cup of coffee", "croissant"]},
    {"id": "comp_04", "prompt": "a bicycle leaning against a brick wall", "attributes": ["bicycle", "brick wall"]},
    {"id": "comp_05", "prompt": "a bird perched on a tree branch", "attributes": ["bird", "tree branch"]},
    {"id": "comp_06", "prompt": "a laptop on a wooden desk with a plant", "attributes": ["laptop", "wooden desk", "plant"]},
    {"id": "comp_07", "prompt": "a guitar standing next to an amplifier", "attributes": ["guitar", "amplifier"]},
    {"id": "comp_08", "prompt": "a teddy bear sitting on a bed", "attributes": ["teddy bear", "bed"]},
    {"id": "comp_09", "prompt": "a camera on a tripod", "attributes": ["camera", "tripod"]},
    {"id": "comp_10", "prompt": "a bottle of wine and two glasses on a table", "attributes": ["bottle of wine", "glasses", "table"]},
    {"id": "comp_11", "prompt": "a soccer ball and a basketball on grass", "attributes": ["soccer ball", "basketball", "grass"]},
    {"id": "comp_12", "prompt": "a piano in a room with a chandelier", "attributes": ["piano", "chandelier"]},
    {"id": "comp_13", "prompt": "a hat on a coat rack next to an umbrella", "attributes": ["hat", "coat rack", "umbrella"]},
    {"id": "comp_14", "prompt": "a pair of sunglasses on a beach towel", "attributes": ["sunglasses", "beach towel"]},
    {"id": "comp_15", "prompt": "a clock on a wall above a fireplace", "attributes": ["clock", "wall", "fireplace"]},

    # === SPATIAL RELATIONSHIPS (15 prompts) ===
    {"id": "spatial_01", "prompt": "a cat sitting under a table", "attributes": ["cat", "table"]},
    {"id": "spatial_02", "prompt": "a dog jumping over a fence", "attributes": ["dog", "fence"]},
    {"id": "spatial_03", "prompt": "a bird flying above the clouds", "attributes": ["bird", "clouds"]},
    {"id": "spatial_04", "prompt": "a fish swimming below a boat", "attributes": ["fish", "boat"]},
    {"id": "spatial_05", "prompt": "a ball rolling towards a goal", "attributes": ["ball", "goal"]},
    {"id": "spatial_06", "prompt": "a child standing behind a tree", "attributes": ["child", "tree"]},
    {"id": "spatial_07", "prompt": "a car driving between two buildings", "attributes": ["car", "buildings"]},
    {"id": "spatial_08", "prompt": "a plane flying through clouds", "attributes": ["plane", "clouds"]},
    {"id": "spatial_09", "prompt": "a person walking along a beach", "attributes": ["person", "beach"]},
    {"id": "spatial_10", "prompt": "a boat floating on a river", "attributes": ["boat", "river"]},
    {"id": "spatial_11", "prompt": "a mountain rising behind a lake", "attributes": ["mountain", "lake"]},
    {"id": "spatial_12", "prompt": "a rainbow arching over a waterfall", "attributes": ["rainbow", "waterfall"]},
    {"id": "spatial_13", "prompt": "a moon shining above a castle", "attributes": ["moon", "castle"]},
    {"id": "spatial_14", "prompt": "a bridge crossing over a canyon", "attributes": ["bridge", "canyon"]},
    {"id": "spatial_15", "prompt": "a tunnel going through a mountain", "attributes": ["tunnel", "mountain"]},
]

print(f"Total prompts: {len(DRAWBENCH_PROMPTS)}")
print(f"  Color Binding: 20")
print(f"  Multi-Object: 15")
print(f"  Spatial: 15")

## Load Models

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading PixArt-α (DiT)...")
pipe = PixArtAlphaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
pipe = pipe.to(device)
print(f"✓ PixArt-α loaded on {device}")

print("\nLoading CLIP...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print("✓ CLIP loaded")

## DynaPrompt Implementation

In [ ]:
class DynaPromptAutoReweighter:
    """
    Hybrid DynaPrompt for DiT evaluation.
    Simplified version for batch evaluation.
    """
    
    def __init__(self, clip_model, clip_preprocess, device,
                 boost_base: float = 1.3, max_iterations: int = 3, 
                 confidence_threshold: float = 0.23):
        self.clip_model = clip_model
        self.clip_preprocess = clip_preprocess
        self.device = device
        self.boost_base = boost_base
        self.max_iterations = max_iterations
        self.confidence_threshold = confidence_threshold
        self.ignore_words = {"a","an","the","of","in","on","at","to","for","with","by","from","is","are","was","were","and","or"}
        
    def extract_critical_words(self, prompt: str) -> List[str]:
        """Extract critical words from prompt."""
        tokens = [w.strip('.,!?;:').lower() for w in prompt.split()]
        return [w for w in tokens if w and w not in self.ignore_words and len(w) > 2]
    
    def analyze_image_for_word(self, image: Image.Image, word: str) -> float:
        """Use CLIP to check if word is present in image."""
        image_input = self.clip_preprocess(image).unsqueeze(0).to(self.device)
        
        text_queries = [
            f"a photo with {word}",
            f"a photo without {word}",
        ]
        text_tokens = clip.tokenize(text_queries).to(self.device)
        
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image_input)
            text_features = self.clip_model.encode_text(text_tokens)
            
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            
        return similarity[0][0].item()
    
    def detect_missed_words(self, image: Image.Image, prompt: str) -> List[Tuple[str, float]]:
        """Detect which words are missing in the image."""
        critical_words = self.extract_critical_words(prompt)
        missed = []
        
        for word in critical_words:
            confidence = self.analyze_image_for_word(image, word)
            if confidence < self.confidence_threshold:
                missed.append((word, confidence))
        
        return missed
    
    def generate_with_dynaprompt(self, prompt: str, pipe, num_inference_steps: int = 25,
                                 guidance_scale: float = 7.5, seed: int = None) -> Tuple[Image.Image, Dict]:
        """Generate with DynaPrompt (simplified for speed)."""
        generator = torch.Generator(device=self.device).manual_seed(seed) if seed else None
        
        metrics = {
            'iterations': 0,
            'final_missed': [],
        }
        
        # Single generation with CLIP checking
        for iteration in range(self.max_iterations):
            metrics['iterations'] = iteration + 1
            
            # Generate
            image = pipe(
                prompt=prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=generator
            ).images[0]
            
            # Check with CLIP
            missed = self.detect_missed_words(image, prompt)
            metrics['final_missed'] = missed
            
            # If all concepts detected, stop early
            if not missed:
                break
        
        return image, metrics

# Initialize reweighter
reweighter = DynaPromptAutoReweighter(
    clip_model=clip_model,
    clip_preprocess=clip_preprocess,
    device=device,
    boost_base=1.3,
    max_iterations=3,
    confidence_threshold=0.23,
)
print("✓ DynaPrompt ready")

## Helper Functions

In [ ]:
def compute_clip_scores(image: Image.Image, attributes: List[str]) -> Dict[str, float]:
    """Compute CLIP score for each attribute."""
    scores = {}
    image_input = clip_preprocess(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        for attr in attributes:
            text_input = clip.tokenize([f"a photo of {attr}"]).to(device)
            text_features = clip_model.encode_text(text_input)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            
            score = (image_features @ text_features.T).item()
            scores[attr] = score
    
    return scores

## Run Evaluation

**Warning**: This will take ~60-90 minutes for 50 prompts (2 images per prompt)

In [ ]:
# Create output directories
output_dir = Path("/content/drawbench_dit")  # Colab path
output_dir.mkdir(parents=True, exist_ok=True)

vanilla_dir = output_dir / "vanilla"
dynaprompt_dir = output_dir / "dynaprompt"
vanilla_dir.mkdir(exist_ok=True)
dynaprompt_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Vanilla images: {vanilla_dir}")
print(f"DynaPrompt images: {dynaprompt_dir}")

In [ ]:
# Results storage
results = {
    "metadata": {
        "model": "PixArt-α XL-2-1024-MS (DiT)",
        "method": "Hybrid DynaPrompt",
        "date": datetime.now().isoformat(),
        "num_prompts": len(DRAWBENCH_PROMPTS),
        "clip_threshold": 0.25,
    },
    "prompts": [],
}

# Category tracking
category_scores = {
    "color": {"vanilla": [], "dynaprompt": []},
    "comp": {"vanilla": [], "dynaprompt": []},
    "spatial": {"vanilla": [], "dynaprompt": []},
}

start_time = time.time()
seed = 42

print("="*80)
print("Starting DrawBench Evaluation")
print("="*80)

for i, prompt_data in enumerate(DRAWBENCH_PROMPTS):
    prompt_id = prompt_data["id"]
    prompt = prompt_data["prompt"]
    attributes = prompt_data["attributes"]
    category = prompt_id.split("_")[0]
    
    print(f"\n[{i+1}/{len(DRAWBENCH_PROMPTS)}] {prompt_id}")
    print(f"  {prompt}")
    
    try:
        # VANILLA
        print("  🎨 Vanilla...", end=" ")
        vanilla_img = pipe(
            prompt=prompt,
            num_inference_steps=20,
            guidance_scale=4.5,
            generator=torch.Generator(device=device).manual_seed(seed + i)
        ).images[0]
        
        vanilla_path = vanilla_dir / f"{prompt_id}.png"
        vanilla_img.save(str(vanilla_path))
        
        vanilla_scores = compute_clip_scores(vanilla_img, attributes)
        vanilla_avg = np.mean(list(vanilla_scores.values()))
        vanilla_passed = all(s >= 0.25 for s in vanilla_scores.values())
        
        print(f"{vanilla_avg:.3f} {'✓' if vanilla_passed else '✗'}")
        
        # DYNAPROMPT
        print("  ✨ DynaPrompt...", end=" ")
        dynaprompt_img, metrics = reweighter.generate_with_dynaprompt(
            prompt=prompt,
            pipe=pipe,
            num_inference_steps=25,
            guidance_scale=7.5,
            seed=seed + i
        )
        
        dynaprompt_path = dynaprompt_dir / f"{prompt_id}.png"
        dynaprompt_img.save(str(dynaprompt_path))
        
        dynaprompt_scores = compute_clip_scores(dynaprompt_img, attributes)
        dynaprompt_avg = np.mean(list(dynaprompt_scores.values()))
        dynaprompt_passed = all(s >= 0.25 for s in dynaprompt_scores.values())
        
        print(f"{dynaprompt_avg:.3f} {'✓' if dynaprompt_passed else '✗'} ({dynaprompt_avg - vanilla_avg:+.3f})")
        
        # Record
        result = {
            "id": prompt_id,
            "prompt": prompt,
            "attributes": attributes,
            "vanilla": {
                "scores": vanilla_scores,
                "avg_score": vanilla_avg,
                "passed": vanilla_passed,
            },
            "dynaprompt": {
                "scores": dynaprompt_scores,
                "avg_score": dynaprompt_avg,
                "passed": dynaprompt_passed,
                "iterations": metrics['iterations'],
            },
            "improvement": dynaprompt_avg - vanilla_avg
        }
        results["prompts"].append(result)
        
        category_scores[category]["vanilla"].append(vanilla_avg)
        category_scores[category]["dynaprompt"].append(dynaprompt_avg)
        
    except Exception as e:
        print(f"  ERROR: {str(e)}")
        results["prompts"].append({
            "id": prompt_id,
            "prompt": prompt,
            "error": str(e),
        })

elapsed_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"✓ Evaluation complete in {elapsed_time/60:.1f} minutes")
print(f"{'='*80}")

## Compute Summary Statistics

In [ ]:
# Calculate summary
vanilla_scores = [p["vanilla"]["avg_score"] for p in results["prompts"] if "vanilla" in p]
dynaprompt_scores = [p["dynaprompt"]["avg_score"] for p in results["prompts"] if "dynaprompt" in p]
vanilla_passed = [p["vanilla"]["passed"] for p in results["prompts"] if "vanilla" in p]
dynaprompt_passed = [p["dynaprompt"]["passed"] for p in results["prompts"] if "dynaprompt" in p]
improvements = [p["improvement"] for p in results["prompts"] if "improvement" in p]

results["summary"] = {
    "total_prompts": len(DRAWBENCH_PROMPTS),
    "successful_generations": len(vanilla_scores),
    "vanilla": {
        "avg_clip_score": np.mean(vanilla_scores),
        "pass_rate": sum(vanilla_passed) / len(vanilla_passed) * 100,
        "total_passed": sum(vanilla_passed),
    },
    "dynaprompt": {
        "avg_clip_score": np.mean(dynaprompt_scores),
        "pass_rate": sum(dynaprompt_passed) / len(dynaprompt_passed) * 100,
        "total_passed": sum(dynaprompt_passed),
    },
    "improvement": {
        "avg_clip_improvement": np.mean(improvements),
        "prompts_improved": sum(1 for imp in improvements if imp > 0),
        "prompts_regressed": sum(1 for imp in improvements if imp < 0),
    },
    "category_scores": {
        "color_binding": {
            "vanilla_avg": np.mean(category_scores["color"]["vanilla"]),
            "dynaprompt_avg": np.mean(category_scores["color"]["dynaprompt"]),
            "count": len(category_scores["color"]["vanilla"]),
        },
        "multi_object": {
            "vanilla_avg": np.mean(category_scores["comp"]["vanilla"]),
            "dynaprompt_avg": np.mean(category_scores["comp"]["dynaprompt"]),
            "count": len(category_scores["comp"]["vanilla"]),
        },
        "spatial": {
            "vanilla_avg": np.mean(category_scores["spatial"]["vanilla"]),
            "dynaprompt_avg": np.mean(category_scores["spatial"]["dynaprompt"]),
            "count": len(category_scores["spatial"]["vanilla"]),
        },
    },
    "elapsed_time_seconds": elapsed_time,
}

# Save results
results_path = output_dir / "results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {results_path}")

## Display Results

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"\nVanilla PixArt-α:")
print(f"  Avg CLIP score: {results['summary']['vanilla']['avg_clip_score']:.3f}")
print(f"  Passed (≥0.25): {results['summary']['vanilla']['total_passed']}/{results['summary']['total_prompts']}")
print(f"  Pass rate: {results['summary']['vanilla']['pass_rate']:.1f}%")

print(f"\nDynaPrompt:")
print(f"  Avg CLIP score: {results['summary']['dynaprompt']['avg_clip_score']:.3f}")
print(f"  Passed (≥0.25): {results['summary']['dynaprompt']['total_passed']}/{results['summary']['total_prompts']}")
print(f"  Pass rate: {results['summary']['dynaprompt']['pass_rate']:.1f}%")

print(f"\nImprovement:")
print(f"  Avg CLIP: {results['summary']['improvement']['avg_clip_improvement']:+.3f}")
print(f"  Prompts improved: {results['summary']['improvement']['prompts_improved']}")
print(f"  Prompts regressed: {results['summary']['improvement']['prompts_regressed']}")

print(f"\nBy Category:")
for cat_name, cat_key in [("Color Binding", "color_binding"), ("Multi-Object", "multi_object"), ("Spatial", "spatial")]:
    cat_data = results['summary']['category_scores'][cat_key]
    print(f"  {cat_name:15s}: {cat_data['vanilla_avg']:.3f} → {cat_data['dynaprompt_avg']:.3f} "
          f"({cat_data['dynaprompt_avg'] - cat_data['vanilla_avg']:+.3f})")

print(f"\nTime: {elapsed_time/60:.1f} minutes")
print("="*80)

## Visualize Results

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall scores by category
ax1 = axes[0]
categories = ['Color\nBinding', 'Multi-\nObject', 'Spatial']
vanilla_avgs = [
    results['summary']['category_scores']['color_binding']['vanilla_avg'],
    results['summary']['category_scores']['multi_object']['vanilla_avg'],
    results['summary']['category_scores']['spatial']['vanilla_avg']
]
dynaprompt_avgs = [
    results['summary']['category_scores']['color_binding']['dynaprompt_avg'],
    results['summary']['category_scores']['multi_object']['dynaprompt_avg'],
    results['summary']['category_scores']['spatial']['dynaprompt_avg']
]

x = np.arange(len(categories))
width = 0.35

ax1.bar(x - width/2, vanilla_avgs, width, label='Vanilla', color='#ff6b6b', alpha=0.8)
ax1.bar(x + width/2, dynaprompt_avgs, width, label='DynaPrompt', color='#4ecdc4', alpha=0.8)
ax1.set_ylabel('Average CLIP Score', fontsize=12)
ax1.set_title('Average CLIP Scores by Category', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Pass rates
ax2 = axes[1]
methods = ['Vanilla', 'DynaPrompt']
pass_rates = [
    results['summary']['vanilla']['pass_rate'],
    results['summary']['dynaprompt']['pass_rate']
]
colors = ['#ff6b6b', '#4ecdc4']

bars = ax2.bar(methods, pass_rates, color=colors, alpha=0.8)
ax2.set_ylabel('Pass Rate (%)', fontsize=12)
ax2.set_title('Pass Rate (CLIP ≥ 0.25 for all attributes)', fontsize=14, fontweight='bold')
ax2.set_ylim([0, 100])
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, rate in zip(bars, pass_rates):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{rate:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / 'evaluation_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to {output_dir / 'evaluation_summary.png'}")

## Download Results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/DrawBench_DiT_Results
!cp -r /content/drawbench_dit/* /content/drive/MyDrive/DrawBench_DiT_Results/

print("✓ Results saved to Google Drive: MyDrive/DrawBench_DiT_Results/")